# Chapter 7 — Part II

This second part concludes Chapter 7 of Friedland by generating Exhibits III and IV.

In [0]:
import chainladder as cl
from IPython.display import display, HTML
import numpy as np
import pandas as pd
import warnings

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

warnings.filterwarnings(
    "ignore",
    message=r"Some exclusions have been ignored\..*link ratio\(s\) is required.*",
    category=UserWarning,
)


## Exhibit III — U.S. Industry P.P. Auto (Impact of Changing Conditions)

In order to illustrate the assumptions and corresponding limitations of the Development method, Exhibit III applies the method to the **U.S. Private Passenger Industry Auto** (USPP Auto) example under four different scenarios:

1. stable claim ratios and no change in case outstanding strength (steady state);
1. increasing claim ratios but no change in case outstanding strength;
1. stable claim ratios but increasing case outstanding strength; and
1. increasing claim ratios and increasing case outstanding strength.

The key assumptions of this study are as follows:

- the earned premium for the first year (1999) is assumed to be $1 million with a 5% annual premium trend;
- (discuss assumed claim ratios)

Since the USPP Auto dataset is contained in the `chainladder` package, we simply load it into a triangle as follows:

In [0]:
triangles = cl.load_sample("friedland_uspp")

For the sheets depicting the claim triangles and age-to-age factors, we reuse the function `dev_exhibit` used in Part I:

In [0]:
def dev_exhibit(tri: cl.Triangle, avg_params: dict[str, int], selected_avg: str, tail: float) -> dict[cl.Triangle()]:
    display('')
    display(HTML("""
    <h2 style='text-align:left;'>
    PART 1 - Data Triangle
    </h2>
    """))
    display(tri)
    display(HTML("""
    <h2 style='text-align:left;'>
    PART 2 - Age-to-Age Factors
    </h2>
    """))
    age_to_age_df = tri.age_to_age.to_frame(origin_as_datetime=False)
    display(
        age_to_age_df.style.format(precision=3, na_rep="")
    )
    devs = {}
    display(HTML("""
    <h2 style='text-align:left;'>
    PART 3 - Average Age-to-Age Factor
    </h2>
    """))
    for k, v in avg_params.items():
        devs[k] = cl.Development(**v).fit_transform(tri)
    def print_ldfs(ldf_dict: dict[cl.Triangle()]):
        with pd.option_context("display.float_format", "{:.3f}".format):
            display(pd.concat([v.to_frame().rename(index={'(All)': k}) for k, v in ldf_dict.items()]))
        return None
    print_ldfs({k: v.ldf_.round(decimals=3) for k,v in devs.items()})
    devs["Selected"] = cl.TailConstant(tail=tail, projection_period=0).fit_transform(devs[selected_avg])
    selected = {}
    selected['CDF to Ultimate'] = devs["Selected"].ldf_.round(decimals=3).incr_to_cum().round(decimals=3)
    selected['Percent Reported'] = (1 / selected['CDF to Ultimate']).round(decimals=3)
    display(HTML("""
    <h2 style='text-align:left;'>
    PART 4 - Selected Age-to-Age Factor
    </h2>
    """))
    print_ldfs({'Selected':devs['Selected'].ldf_.round(decimals=3)})
    print_ldfs(selected)
    return devs

We also define a special formatting function `ex_sht1` in order to create Sheet 1 of Exhibit III, which has its own unique layout:

In [0]:
def ex3sht1(
          tri: cl.Triangle,
          dev_input: dict,
          tail_input: dict
) -> tuple:
    
    def format_col(x, reverse = False):
            if reverse is True:
                return x.to_frame().squeeze()[::-1].reset_index(drop=True)
            return x.to_frame().squeeze().reset_index(drop=True)

    def col_diff(x, y):
        return x.squeeze() - y.squeeze()

    def col_div(x, y):
        return x.squeeze() / y.squeeze()

    def col_mult(x, y):
        return x.squeeze() * y.squeeze()

    # scenario = "Steady State"

    dev = cl.Development(**dev_input).fit(tri)
    model = cl.Chainladder().fit(dev.transform(tri))
    ult = model.ultimate_

    col1 = pd.DataFrame({"Accident Year": format_col(tri.origin.astype(str))})
    col2 = pd.DataFrame({"Earned Premium": format_col(tri.latest_diagonal.loc["Steady State", "Earned Premium"])})
    col4 = pd.DataFrame({"Ult. Claims": format_col(ult.loc["Steady State", "Reported Claims"])})
    col3 = pd.DataFrame({"Ult. Claim Ratio": col_div(col4, col2)})
    col5 = pd.DataFrame({"Rep. Claims 12/31/08": format_col(tri.latest_diagonal.loc["Steady State", "Reported Claims"])})
    col6 = pd.DataFrame({"Actual IBNR": col_diff(col4, col5)})
    col8 = pd.DataFrame({"Ult. Claims": format_col(ult.loc["Increasing Claim", "Reported Claims"])})
    col9 = pd.DataFrame({"Rep. Claims 12/31/08": format_col(tri.latest_diagonal.loc["Increasing Claim", "Reported Claims"])})
    col7 = pd.DataFrame({"Ult. Claim Ratio": col_div(col8, col2)})
    col10 = pd.DataFrame({"Actual IBNR": col_diff(col8, col9)})
    col11 = pd.DataFrame({"Accident Year": format_col(tri.origin.astype(str))})
    col12 = pd.DataFrame({"Earned Premium": format_col(tri.latest_diagonal.loc["Steady State", "Earned Premium"])})
    col14 = pd.DataFrame({"Ult. Claims": format_col(ult.loc["Steady State", "Reported Claims"])})
    col13 = pd.DataFrame({"Ult. Claim Ratio": col_div(col14, col12)})
    col15 = pd.DataFrame({"Rep. Claims 12/31/08": format_col(tri.latest_diagonal.loc["Increasing Case", "Reported Claims"])})
    col16 = pd.DataFrame({"Actual IBNR": col_diff(col14, col15)})
    col18 = pd.DataFrame({"Ult. Claims": format_col(ult.loc["Increasing Claim", "Reported Claims"])})
    col17 = pd.DataFrame({"Ult. Claim Ratio": col_div(col18, col12)})
    col19 = pd.DataFrame({"Rep. Claims 12/31/08": format_col(tri.latest_diagonal.loc["Increasing Claim Case", "Reported Claims"])})
    col20 = pd.DataFrame({"Actual IBNR": col_diff(col18, col19)})

    cols = [col1, col2, col3, col4, col5, col6]
    df1 = pd.concat([col1, col2], axis=1)
    df2 = pd.concat([col3, col4, col5, col6], axis=1)
    df3 = pd.concat([col7, col8, col9, col10], axis=1)
    df4 = pd.concat([col11, col12], axis=1)
    df5 = pd.concat([col13, col14, col15, col16], axis=1)
    df6 = pd.concat([col17, col18, col19, col20], axis=1)

    dfs_upper = [df1, df2, df3]
    dfs_lower = [df4, df5, df6]

    results_upper = pd.concat(dfs_upper, axis=1, keys=["", "Steady State", "Increasing Claim Ratios"])
    results_upper = results_upper.set_index(("", "Accident Year")) # since its multi-index, I needed to use the tuple to designate the index col
    results_upper.index.name = "Accident Year" 
    results_upper.loc["Total"] = results_upper.sum()

    # remove the totals from the ratio columns (where totals do not make sense)
    results_upper.loc["Total", ("Steady State", "Ult. Claim Ratio")] = np.nan
    results_upper.loc["Total", ("Increasing Claim Ratios", "Ult. Claim Ratio")] = np.nan

    results_lower = pd.concat(dfs_lower, axis=1, keys=["", "Increasing Case Outstanding Strength", "Increasing Claim Ratios and Case Outstanding Strength"])
    results_lower = results_lower.set_index(("", "Accident Year")) # since its multi-index, I needed to use the tuple to designate the index col
    results_lower.index.name = "Accident Year" 
    results_lower.loc["Total"] = results_lower.sum()

    # remove the totals from the ratio columns (where totals do not make sense)
    results_lower.loc["Total", ("Increasing Case Outstanding Strength", "Ult. Claim Ratio")] = np.nan
    results_lower.loc["Total", ("Increasing Claim Ratios and Case Outstanding Strength", "Ult. Claim Ratio")] = np.nan

    display(HTML("""
    <h2 style='text-align:center;'>
    Exhibit III Sheet 1: Summary of Earned Premium and Claim Ratio Assumptions and Actual IBNR
    </h2>
    """))
    
    upper_formats = {
        ("Steady State", "Ult. Claim Ratio"): "{:.1%}",
        ("Steady State", "Ult. Claims"): "{:,.0f}",
        ("Steady State", "Rep. Claims 12/31/08"): "{:,.0f}",
        ("Steady State", "Actual IBNR"): "{:,.0f}",

        ("Increasing Claim Ratios", "Ult. Claim Ratio"): "{:.1%}",
        ("Increasing Claim Ratios", "Ult. Claims"): "{:,.0f}",
        ("Increasing Claim Ratios", "Rep. Claims 12/31/08"): "{:,.0f}",
        ("Increasing Claim Ratios", "Actual IBNR"): "{:,.0f}",

        ("", "Earned Premium"): "{:,.0f}",
    }

    display(
        results_upper.style.format(
             upper_formats,
             na_rep=""
        )
    )
    
    lower_formats = {
        ("Increasing Case Outstanding Strength", "Ult. Claim Ratio"): "{:.1%}",
        ("Increasing Case Outstanding Strength", "Ult. Claims"): "{:,.0f}",
        ("Increasing Case Outstanding Strength", "Rep. Claims 12/31/08"): "{:,.0f}",
        ("Increasing Case Outstanding Strength", "Actual IBNR"): "{:,.0f}",

        (
            "Increasing Claim Ratios and Case Outstanding Strength",
            "Ult. Claim Ratio",
        ): "{:.1%}",
        (
            "Increasing Claim Ratios and Case Outstanding Strength",
            "Ult. Claims",
        ): "{:,.0f}",
        (
            "Increasing Claim Ratios and Case Outstanding Strength",
            "Rep. Claims 12/31/08",
        ): "{:,.0f}",
        (
            "Increasing Claim Ratios and Case Outstanding Strength",
            "Actual IBNR",
        ): "{:,.0f}",

        ("", "Earned Premium"): "{:,.0f}",
    }

    display(
         results_lower.style.format(
              lower_formats,
              na_rep=""
            )
    )

    return (results_upper, results_lower)

### Exhibit III Sheet 1: Summary of Earned Premium and Claim Ratio Assumptions and Actual IBNR

We begin by generating Exhibit III, Sheet 1:

In [0]:
dev_input = {
    "average": "volume",
    "n_periods": 5
}

tail_input = {
    "tail": 1.0,
    "projection_period": -1
}

res_up, res_low = ex3sht1(
    triangles,
    dev_input,
    tail_input,
)


In [0]:
assert np.allclose(
    res_up.loc[
        res_up.index != "Total",
        ("Steady State", ["Ult. Claims", "Actual IBNR"]),
    ].values,
    np.array([
        [700000, 0],
        [735000, 0],
        [771750, 0],
        [810338, 0],
        [850854, 8509],
        [893397, 8934],
        [938067, 18761],
        [984970, 49249],
        [1034219, 103422],
        [1085930, 249764],
    ]),
    atol=1,
    rtol=0,
)

assert np.allclose(
    res_up.loc[
        res_up.index != "Total",
        ("Increasing Claim Ratios", ["Ult. Claims", "Actual IBNR"]),
    ].values,
    np.array([
        [700000, 0],
        [735000, 0],
        [771750, 0],
        [810338, 0],
        [850854, 8509],
        [1021025, 10210],
        [1139081, 22782],
        [1266390, 63320],
        [1403583, 140358],
        [1551328, 356805],
    ]),
    atol=1,
    rtol=0,
)

assert np.allclose(
    res_low.loc[
        res_low.index != "Total",
        ("Increasing Case Outstanding Strength", ["Ult. Claims", "Actual IBNR"]),
    ].values,
    np.array([
        [700000, 0],
        [735000, 0],
        [771750, 0],
        [810338, 0],
        [850854, 8509],
        [893397, 8934],
        [938067, 4690],
        [984970, 22162],
        [1034219, 54296],
        [1085930, 154745],
    ]),
    atol=2,
    rtol=0,
)

assert np.allclose(
    res_low.loc[
        res_low.index != "Total",
        (
            "Increasing Claim Ratios and Case Outstanding Strength",
            ["Ult. Claims", "Actual IBNR"],
        ),
    ].values,
    np.array([
        [700000, 0],
        [735000, 0],
        [771750, 0],
        [810338, 0],
        [850854, 8509],
        [1021025, 10210],
        [1139081, 5695],
        [1266390, 28494],
        [1403583, 73688],
        [1551328, 221064],
    ]),
    atol=1,
    rtol=0,
)

### Exhibit III, Sheet 2 - Sheet 2: USPP Auto Steady-State - Reported Claims

We then generate Sheet 2, Parts 1-4 which contain the data triangle and age-to-age factors for USPP reported claims under the steady-state scenario (p. 115).

In [0]:
devs = dev_exhibit(
    triangles.loc["Steady State", "Reported Claims"],
    avg_params={"volume_5": {'n_periods': 5, 'average': 'volume'}},
    selected_avg='volume_5', tail=1
    )

In [0]:
assert np.allclose(
    devs["Selected"].ldf_.round(3).values.squeeze(),
    np.array([
        1.169, 1.056, 1.032, 1.010, 1.000,
        1.010, 1.000, 1.000, 1.000, 1.000,
    ]),
    atol=0.0005,
    rtol=0,
)

### Exhibit III, Sheet 3: USPP Auto Steady-State - Paid Claims

We then generate Sheet 3, Parts 1-4 which contain the data triangle and age-to-age factors for USPP paid claims under the steady-state scenario (p. 116).

In [0]:
devs = dev_exhibit(
    triangles.loc["Steady State", "Paid Claims"],
    avg_params={"volume_5": {'n_periods': 5, 'average': 'volume'}},
    selected_avg='volume_5',
    tail=1
    )

In [0]:
assert np.allclose(
    devs["Selected"].ldf_.round(3).values.squeeze(),
    np.array([
        1.690, 1.183, 1.095, 1.043, 1.021,
        1.010, 1.000, 1.010, 1.000, 1.000,
    ]),
    atol=0.0005,
    rtol=0,
)

### Exhibit III, Sheet 4: USPP Auto Increasing Claim Ratios - Reported Claims

We then generate Sheet 4, Parts 1-4 which contain the data triangle and age-to-age factors for USPP reported claims under the increasing claim ratio scenario (p. 117).

In [0]:
devs = dev_exhibit(
    triangles.loc["Increasing Claim", "Reported Claims"],
    avg_params={"volume_5": {'n_periods': 5, 'average': 'volume'}},
    selected_avg='volume_5',
    tail=1
    )

In [0]:
assert np.allclose(
    devs["Selected"].ldf_.round(3).values.squeeze(),
    np.array([
        1.169, 1.056, 1.032, 1.010, 1.000,
        1.010, 1.000, 1.000, 1.000, 1.000,
    ]),
    atol=0.0005,
    rtol=0,
)

### Exhibit III, Sheet 5: USPP Auto Increasing Claim Ratios - Paid Claims

We then generate Sheet 5, Parts 1-4 which contain the data triangle and age-to-age factors for USPP paid claims under the increasing claim ratio scenario (p. 118).

In [0]:
devs = dev_exhibit(
    triangles.loc["Increasing Claim", "Paid Claims"],
    avg_params={"volume_5": {'n_periods': 5, 'average': 'volume'}},
    selected_avg='volume_5',
    tail=1
    )

In [0]:
assert np.allclose(
    devs["Selected"].ldf_.round(3).values.squeeze(),
    np.array([
        1.690, 1.183, 1.095, 1.043, 1.021,
        1.010, 1.000, 1.010, 1.000, 1.000,
    ]),
    atol=0.0005,
    rtol=0,
)

### Exhibit III, Sheet 6: USPP Auto Increasing Case Outstanding Strength - Reported Claims

We then generate Sheet 6, Parts 1-4 which contain the data triangle and age-to-age factors for USPP reported claims under the increasing case outstanding strength scenario (p. 119).

In [0]:
devs = dev_exhibit(
    triangles.loc["Increasing Case", "Reported Claims"],
    avg_params={"volume_5": {'n_periods': 5, 'average': 'volume'}},
    selected_avg='volume_5',
    tail=1
    )

In [0]:
assert np.allclose(
    devs["Selected"].ldf_.round(3).values.squeeze(),
    np.array([
        1.178, 1.061, 1.034, 1.009, 1.000,
        1.010, 1.000, 1.000, 1.000, 1.000,
    ]),
    atol=0.0005,
    rtol=0,
)

### Exhibit III, Sheet 7: USPP Auto Increasing Case Outstanding Strength - Paid Claims

We then generate Sheet 7, Parts 1-4 which contain the data triangle and age-to-age factors for USPP paid claims under the increasing case outstanding strength scenario (p. 120).

In [0]:
devs = dev_exhibit(
    triangles.loc["Increasing Case", "Paid Claims"],
    avg_params={"volume_5": {'n_periods': 5, 'average': 'volume'}},
    selected_avg='volume_5',
    tail=1
    )

In [0]:
assert np.allclose(
    devs["Selected"].ldf_.round(3).values.squeeze(),
    np.array([
        1.690, 1.183, 1.095, 1.043, 1.021,
        1.010, 1.000, 1.010, 1.000, 1.000,
    ]),
    atol=0.0005,
    rtol=0,
)

### Exhibit III, Sheet 8: USPP Auto Increasing Claim Ratios and Increasing Case Outstanding Strength - Reported Claims

We then generate Sheet 8, Parts 1-4 which contain the data triangle and age-to-age factors for USPP reported claims under the increasing claims ratio / increasing case outstanding strength scenario (p. 121).

In [0]:
devs = dev_exhibit(
    triangles.loc["Increasing Claim Case", "Reported Claims"],
    avg_params={"volume_5": {'n_periods': 5, 'average': 'volume'}},
    selected_avg='volume_5',
    tail=1
    )

In [0]:
assert np.allclose(
    devs["Selected"].ldf_.round(3).values.squeeze(),
    np.array([
        1.179, 1.061, 1.035, 1.009, 1.000,
        1.010, 1.000, 1.000, 1.000, 1.000,
    ]),
    atol=0.0005,
    rtol=0,
)

### Exhibit III, Sheet 9: USPP Auto Increasing Claim Ratios and Increasing Case Outstanding Strength - Paid Claims

We then generate Sheet 9, Parts 1-4 which contain the data triangle and age-to-age factors for USPP paid claims under the increasing claims ratio / increasing case outstanding strength scenario (p. 122):

In [0]:
devs = dev_exhibit(
    triangles.loc["Increasing Claim Case", "Paid Claims"],
    avg_params={"volume_5": {'n_periods': 5, 'average': 'volume'}},
    selected_avg='volume_5',
    tail=1
    )

In [0]:
assert np.allclose(
    devs["Selected"].ldf_.round(3).values.squeeze(),
    np.array([
        1.690, 1.183, 1.095, 1.043, 1.021,
        1.010, 1.000, 1.010, 1.000, 1.000,
    ]),
    atol=0.0005,
    rtol=0,
)

### Exhibit III, Sheets 10 and 11: USPP Auto Development of Unpaid Claim Estimate

We first create a function to handle formatting as follows:

In [0]:
def Ex3Sht10(scenario, tr):

    def format_col(x, reverse = False):
        if reverse is True:
            return x.to_frame().squeeze()[::-1].reset_index(drop=True)
        return x.to_frame().squeeze().reset_index(drop=True)

    def col_diff(x, y):
        return x.squeeze() - y.squeeze()

    def col_mult(x, y):
        return x.squeeze() * y.squeeze()

    tri = tr.copy()

    dev_input = {
        "average": "volume",
        "n_periods": 5
    }

    tail_input = {
        "tail": 1.0,
        "projection_period": -1
    }

    dev = cl.Development(**dev_input)
    tail = cl.TailConstant(**tail_input)
    tri = tail.fit_transform(dev.fit_transform(tri))

    actual_scenario = (
    "Steady State"
    if scenario in ["Steady State", "Increasing Case"]
    else "Increasing Claim"
    )

    cdf = format_col(tri.cdf_.loc[actual_scenario]["Reported Claims"], reverse=True)
    ult = format_col(tri.loc[actual_scenario]["Reported Claims"].latest_diagonal)
    
    col1 = pd.DataFrame({
        "Accident Year": format_col(tr.origin)
        })

    col2 = pd.DataFrame({
        "Age of Accident Year at 12/31/08": format_col(tr.development, reverse=True)

    })

    col3 = pd.DataFrame({
        "Claims at 12/31/2008 - Reported": format_col(tri.loc[scenario]["Reported Claims"].latest_diagonal)

    })

    col4 = pd.DataFrame({
        "Claims at 12/31/2008 - Paid": format_col(tri.loc[scenario]["Paid Claims"].latest_diagonal)

    })

    col5 = pd.DataFrame({
        "Case Outstanding": col_diff(col3, col4)

    })

    col6 = pd.DataFrame({
        "CDF to Ult. - Reported": format_col(tri.cdf_.loc[scenario]["Reported Claims"], reverse=True)

    })

    col7 = pd.DataFrame({
        "CDF to Ult. - Paid": format_col(tri.cdf_.loc[scenario]["Paid Claims"], reverse=True)
    })

    col8 = pd.DataFrame({
        "Projected Ult. Claims Using Dev. Method - Reported": col_mult(col3, col6)
    })

    col9 = pd.DataFrame({
        "Projected Ult. Claims Using Dev. Method - Paid": col_mult(col4, col7)
    })

    col10 = pd.DataFrame({
        "Estimated IBNR Using Dev. Method - Reported": col_diff(col8, col3)
    })

    col11 = pd.DataFrame({
        "Estimated IBNR Using Dev. Method - Paid": col_diff(col9, col3)
    })

    col12 = pd.DataFrame({
        "Actual IBNR": col_diff(col_mult(ult, cdf), col3)
    })

    col13 = pd.DataFrame({
        "Difference from Actual IBNR - Reported": col_diff(col12, col10)
    })

    col14 = pd.DataFrame({
        "Difference from Actual IBNR - Paid": col_diff(col12, col11)
    })


    cols = [col1, col2, col3, col4, col5, col6, col7, col8, col9, col10, col11, col12, col13, col14]

    results = pd.concat(cols, axis=1)

    format_dict = {
        # Dollar amounts
        "Claims at 12/31/2008 - Reported": "{:,.0f}",
        "Claims at 12/31/2008 - Paid": "{:,.0f}",
        "Case Outstanding": "{:,.0f}",
        "Projected Ult. Claims Using Dev. Method - Reported": "{:,.0f}",
        "Projected Ult. Claims Using Dev. Method - Paid": "{:,.0f}",
        "Estimated IBNR Using Dev. Method - Reported": "{:,.0f}",
        "Estimated IBNR Using Dev. Method - Paid": "{:,.0f}",
        "Actual IBNR": "{:,.0f}",
        "Difference from Actual IBNR - Reported": "{:,.0f}",
        "Difference from Actual IBNR - Paid": "{:,.0f}",

        # Ages
        "Age of Accident Year at 12/31/08": "{:.0f}",

        # Factors
        "CDF to Ult. - Reported": "{:.3f}",
        "CDF to Ult. - Paid": "{:.3f}",
    }

    results = results.set_index("Accident Year")

    total_cols = [
        "Claims at 12/31/2008 - Reported",
        "Claims at 12/31/2008 - Paid",
        "Case Outstanding",
        "Projected Ult. Claims Using Dev. Method - Reported",
        "Projected Ult. Claims Using Dev. Method - Paid",
        "Estimated IBNR Using Dev. Method - Reported",
        "Estimated IBNR Using Dev. Method - Paid",
        "Actual IBNR",
        "Difference from Actual IBNR - Reported",
        "Difference from Actual IBNR - Paid",
    ]
    
    results.loc["Total", total_cols] = results[total_cols].sum()

    display(HTML(f"<h3>Exhibit III Sheets 10 and 11 — {scenario}</h3>"))

    display(
        results.style.format(format_dict, na_rep="")
    )

    return results



We then generate Sheets 10 and 11 which illustrate the differences between the actual IBNR and the IBNR obtained through the development method for each scenario (pp. 123-124).

In [0]:
res_1 = Ex3Sht10("Steady State", triangles)

res_2 = Ex3Sht10("Increasing Claim", triangles)

res_3 = Ex3Sht10("Increasing Case", triangles)

res_4 = Ex3Sht10("Increasing Claim Case", triangles)


In [0]:
estimate_cols = [
    "Projected Ult. Claims Using Dev. Method - Reported",  # (8)
    "Projected Ult. Claims Using Dev. Method - Paid",      # (9)
    "Estimated IBNR Using Dev. Method - Reported",         # (10)
    "Estimated IBNR Using Dev. Method - Paid",             # (11)
]

assert np.allclose(
    res_1.loc[
        res_1.index != "Total",
        estimate_cols,
    ].values,
    np.array([
    [700000,  700000,       0,      0],
    [735000,  735000,       0,      0],
    [771750,  771750,       0,      0],
    [810338,  810338,       0,      0],
    [850854,  850854,    8509,   8509],
    [893397,  893397,    8934,   8934],
    [938067,  938067,   18761,  18761],
    [984970,  984970,   49249,  49249],
    [1034219, 1034219, 103422, 103422],
    [1085930, 1085930, 249764, 249764],
]),
    atol=3,
    rtol=0,
)

assert np.allclose(
    res_2.loc[
        res_2.index != "Total",
        estimate_cols,
    ].values,
    np.array([
    [700000,  700000,       0,      0],
    [735000,  735000,       0,      0],
    [771750,  771750,       0,      0],
    [810338,  810338,       0,      0],
    [850854,  850854,    8509,   8509],
    [1021025, 1021025,  10210,  10210],
    [1139081, 1139081,  22782,  22782],
    [1266390, 1266390,  63320,  63320],
    [1403583, 1403583, 140358, 140358],
    [1551328, 1551328, 356805, 356805],
]),
    atol=3,
    rtol=0,
)

assert np.allclose(
    res_3.loc[
        res_3.index != "Total",
        estimate_cols,
    ].values,
    np.array([
    [700000,  700000,       0,      0],
    [735000,  735000,       0,      0],
    [771750,  771750,       0,      0],
    [810338,  810338,       0,      0],
    [850854,  850854,    8509,   8509],
    [893397,  893397,    8934,   8934],
    [951656,  938067,   18279,   4690],
    [1015302, 984970,   52493,  22162],
    [1096235, 1034219, 116313,  54296],
    [1227589, 1085930, 296404, 154745],
]),
    atol=3,
    rtol=0,
)

assert np.allclose(
    res_4.loc[
        res_4.index != "Total",
        estimate_cols,
    ].values,
    np.array([
    [700000,  700000,       0,      0],
    [735000,  735000,       0,      0],
    [771750,  771750,       0,      0],
    [810338,  810338,       0,      0],
    [850854,  850854,    8509,   8509],
    [1021025, 1021025,  10210,  10210],
    [1155482, 1139081,  22096,   5695],
    [1305639, 1266390,  67742,  28494],
    [1488874, 1403583, 158980,  73688],
    [1756504, 1551328, 426240, 221064],
]),
    atol=3,
    rtol=0,
)


## Exhibit IV — Impact of Changing Product Mix Example

In Exhibit IV, Friedland demonstrates the effect of changes in product mix on the development method in order to further illustrate its limitations before segueing into more advanced methods.  The changing product mix in this example refers to the proportion of earned premiums attributable to the private passenger and commercial portfolios of U.S. Auto over a ten-year experience period. 

In the steady state scenario, both lines have an equal proportion of the total earned premiums (which increase by 5% annually), while in the changing product mix scenario, the proportionate share of earned premiums of commercial lines increases relative to private passenger (specifically, commercial's earned premium increases by 30% annually from the seventh development year). Other than the earned premium, the other key assumption of this study are as follows are assumed ultimate claim ratios of 70% and 80%, respectively, for private passenger and commercial.

Since the USPP Auto dataset is contained in the `chainladder` package, we simply load it into a triangle as follows:

In [0]:
data = cl.load_sample("friedland_us_auto")
tri = data.copy()

### Exhibit IV, Sheet 1 - Summary of Assumptions - Earned Premiums and Claim Ratios

To use Chainladder to reproduce Sheet 1 of Exhibit IV, we create the following function to take care of the necessary formatting, and to perform the appropriate transformations/slicing of the data triangle to apply the development method and determine the IBNR. 

In [0]:
def Ex4Sht1(scenario, tri):

    def format_col(x, reverse = False):
        if reverse is True:
            return x.to_frame().squeeze()[::-1].reset_index(drop=True)
        return x.to_frame().squeeze().reset_index(drop=True)

    def col_add(x, y):
        return x.squeeze() + y.squeeze()

    def col_diff(x, y):
        return x.squeeze() - y.squeeze()

    def col_div(x, y):
        return x.squeeze() / y.squeeze()

    def col_mult(x, y):
        return x.squeeze() * y.squeeze()

    tri = tri.copy()

    dev_input = {
        "average": "volume",
        "n_periods": 5
    }

    tail_input = {
        "tail": 1.0,
        "projection_period": -1
    }

    dev = cl.Development(**dev_input)
    tail = cl.TailConstant(**tail_input)
    tri = tail.fit_transform(dev.fit_transform(tri))

    col1 = pd.DataFrame({
            "Accident Year": format_col(tri.origin)
            })

    col2 = pd.DataFrame({
                "Priv Pass Auto": format_col(tri.loc["Steady State"]["Earned Premium"].latest_diagonal / 2)
                })

    col3 = pd.DataFrame({
            "Comm Auto": col_diff(
                format_col(tri.loc[scenario]["Earned Premium"].latest_diagonal),
                col2
                )
            })

    col4 = pd.DataFrame({
            "Total": format_col(tri.loc[scenario]["Earned Premium"].latest_diagonal)
            })

    col5 = pd.DataFrame({
            "Priv Pass Auto (%)": pd.Series([0.7] * 10) 
            })

    col6 = pd.DataFrame({
            "Comm Auto (%)": pd.Series([0.8] * 10) 
            })

    col8 = pd.DataFrame({
            "Priv Pass Auto": col_mult(col2, col5) 
            })

    col9 = pd.DataFrame({
            "Comm Auto": col_mult(col3, col6) 
            })

    col10 = pd.DataFrame({
            "Comm Auto": col_add(col8, col9) 
            })

    col7 = pd.DataFrame({
            "Total (%)": col_div(col10, col4) 
            })

    col11 = pd.DataFrame({
            "Reported Claims as at 12/31/2008": format_col(tri.loc[scenario]["Reported Claims"].latest_diagonal) 
            })

    col12 = pd.DataFrame({
            "Actual IBNR": col_diff(col10, col11) 
            })


    cols = [col1, col2, col3, col4, col5, col6, col7, col8, col9, col10, col11, col12]

    results = pd.concat(cols, axis=1)

    format_dict = {
        # Dollar amounts
        "Priv Pass Auto": "{:,.0f}",
        "Comm Auto": "{:,.0f}",
        "Total": "{:,.0f}",
        "Reported Claims as at 12/31/2008": "{:,.0f}",
        "Actual IBNR": "{:,.0f}",

        # Percentages
        "Priv Pass Auto (%)": "{:.1%}",
        "Comm Auto (%)": "{:.1%}",
        "Total (%)": "{:.1%}",

        # Factors
        "CDF to Ult. - Reported": "{:.3f}",
        "CDF to Ult. - Paid": "{:.3f}",
    }

    results = results.set_index("Accident Year")

    total_cols = [
        "Priv Pass Auto",
        "Comm Auto",
        "Total",
        "Reported Claims as at 12/31/2008",
        "Actual IBNR",
    ]

    results.loc["Total", total_cols] = results[total_cols].sum()

    display(HTML(f"<h3>Exhibit IV, Sheet 1 — {scenario}</h3>"))

    display(
        results.style.format(format_dict, na_rep="").set_table_styles([
            {
                "selector": "th.col_heading",
                "props": [
                    ("white-space", "normal"),
                    ("max-width", "80px"),
                ]
            }
        ])
    )

    return results

Having defined the function, we generate Sheet 1, which contains the development of losses under the Steady State and Changing Product Mix scenarios (p. 125):

In [0]:
res_1 = Ex4Sht1("Steady State", data)

res_2 = Ex4Sht1("Changing Product Mix", data)


In [0]:
assert np.allclose(
    res_1.iloc[:-1, [8, 10]].values,
    np.array([
        [1500000,       0],
        [1575000,       0],
        [1653750,       0],
        [1736438,       0],
        [1823259,    8509],
        [1914422,   29354],
        [2010143,   61644],
        [2110651,  173073],
        [2216183,  363454],
        [2326992,  758599],
    ]),
    atol=2,
    rtol=0,
)

assert np.allclose(
    res_2.iloc[:-1, [8, 10]].values,
    np.array([
        [1500000,       0],
        [1575000,       0],
        [1653750,       0],
        [1736438,       0],
        [1823259,    8509],
        [1914422,   29354],
        [2265400,   71855],
        [2710503,  239057],
        [3277411,  596924],
        [4002080, 1445385],
    ]),
    atol=2,
    rtol=0,
)

### Exhibit IV, Sheet 2 - U.S. Auto Steady-State -  Reported Claims

We now use the 'dev_exhibit' function to generate Exhibit IV, Sheet 2 (p. 126):

In [0]:
devs = dev_exhibit(
    data.loc["Steady State", "Reported Claims"],
    avg_params={"volume_5": {'n_periods': 5, 'average': 'volume'}},
    selected_avg='volume_5',
    tail=1
    )

In [0]:
assert np.allclose(
    devs["Selected"].ldf_.round(3).values.squeeze(),
    np.array([
    1.240, 1.098, 1.056, 1.016, 1.011,
    1.005, 1.000, 1.000, 1.000, 1.000,
]),
    atol=0.0005,
    rtol=0,
)

### Exhibit IV, Sheet 3 - U.S. Auto Steady-State -  Paid Claims

We now use the 'dev_exhibit' function to generate Exhibit IV, Sheet 3 (p. 127):

In [0]:
devs = dev_exhibit(
    data.loc["Steady State", "Paid Claims"],
    avg_params={"volume_5": {'n_periods': 5, 'average': 'volume'}},
    selected_avg='volume_5',
    tail=1
    )

In [0]:
assert np.allclose(
    devs["Selected"].ldf_.round(3).values.squeeze(),
    np.array([
    1.840, 1.299, 1.157, 1.077, 1.033,
    1.016, 1.005, 1.010, 1.005, 1.000,
]),
    atol=0.0005,
    rtol=0,
)

### Exhibit IV, Sheet 4 - U.S. Auto Changing Product Mix -  Reported Claims

We now use the 'dev_exhibit' function to generate Exhibit IV, Sheet 4 (p. 128):

In [0]:
devs = dev_exhibit(
    data.loc["Changing Product Mix", "Reported Claims"],
    avg_params={"volume_5": {'n_periods': 5, 'average': 'volume'}},
    selected_avg='volume_5',
    tail=1
    )

In [0]:
assert np.allclose(
    devs["Selected"].ldf_.round(3).values.squeeze(),
    np.array([
    1.252, 1.101, 1.057, 1.016, 1.011,
    1.005, 1.000, 1.000, 1.000, 1.000,
]),
    atol=0.0005,
    rtol=0,
)

### Exhibit IV, Sheet 5 - U.S. Auto Changing Product Mix -  Paid Claims

We now use the 'dev_exhibit' function to generate Exhibit IV, Sheet 5 (p. 129):

In [0]:
devs = dev_exhibit(
    data.loc["Changing Product Mix", "Paid Claims"],
    avg_params={"volume_5": {'n_periods': 5, 'average': 'volume'}},
    selected_avg='volume_5',
    tail=1
    )

In [0]:
assert np.allclose(
    devs["Selected"].ldf_.round(3).values.squeeze(),
    np.array([
    1.870, 1.310, 1.158, 1.077, 1.033,
    1.016, 1.005, 1.010, 1.005, 1.000,
]),
    atol=0.0005,
    rtol=0,
)

### Exhibit IV, Sheet 6

Finally, we produce the summary table contained in Exhibit IV, Sheet 6 (p. 130) by defining the following function to handle formatting:

In [0]:
def Ex4Sht6(scenario, tr):

    def format_col(x, reverse = False):
        if reverse is True:
            return x.to_frame().squeeze()[::-1].reset_index(drop=True)
        return x.to_frame().squeeze().reset_index(drop=True)

    def col_diff(x, y):
        return x.squeeze() - y.squeeze()

    def col_mult(x, y):
        return x.squeeze() * y.squeeze()

    def col_sum(x, y):
            return x.squeeze() + y.squeeze()

    tri = tr.copy()

    dev_input = {
        "average": "volume",
        "n_periods": 5
    }

    tail_input = {
        "tail": 1.0,
        "projection_period": -1
    }

    dev = cl.Development(**dev_input)
    tail = cl.TailConstant(**tail_input)
    tri = tail.fit_transform(dev.fit_transform(tri))

    
    col1 = pd.DataFrame({
        "Accident Year": format_col(tr.origin)
        })

    col2 = pd.DataFrame({
        "Age of Accident Year at 12/31/08": format_col(tr.development, reverse=True)

    })

    col3 = pd.DataFrame({
        "Claims at 12/31/2008 - Reported": format_col(tri.loc[scenario]["Reported Claims"].latest_diagonal)

    })

    col4 = pd.DataFrame({
        "Claims at 12/31/2008 - Paid": format_col(tri.loc[scenario]["Paid Claims"].latest_diagonal)

    })

    col5 = pd.DataFrame({
        "Case Outstanding": col_diff(col3, col4)

    })

    col6 = pd.DataFrame({
        "CDF to Ult. - Reported": format_col(tri.cdf_.loc[scenario]["Reported Claims"], reverse=True)

    })

    col7 = pd.DataFrame({
        "CDF to Ult. - Paid": format_col(tri.cdf_.loc[scenario]["Paid Claims"], reverse=True)
    })

    col8 = pd.DataFrame({
        "Projected Ult. Claims Using Dev. Method - Reported": col_mult(col3, col6)
    })

    col9 = pd.DataFrame({
        "Projected Ult. Claims Using Dev. Method - Paid": col_mult(col4, col7)
    })

    col10 = pd.DataFrame({
        "Estimated IBNR Using Dev. Method - Reported": col_diff(col8, col3)
    })

    col11 = pd.DataFrame({
        "Estimated IBNR Using Dev. Method - Paid": col_diff(col9, col3)
    })

    colA = col_mult(
        format_col(tri.loc["Steady State"]["Earned Premium"].latest_diagonal / 2),
        pd.Series([0.7] * 10)
        )
    colB = col_diff(
        format_col(tri.loc[scenario]["Earned Premium"].latest_diagonal), 
        format_col(tri.loc["Steady State"]["Earned Premium"].latest_diagonal / 2)
        ) * pd.Series([0.8] * 10)
    colC =  format_col(tri.loc[scenario]["Reported Claims"].latest_diagonal)
    colD = col_diff(
        col_sum(
            colA, 
            colB
            ), 
            colC)

    col12 = pd.DataFrame({
        "Actual IBNR": format_col(colD) 
    })

    col13 = pd.DataFrame({
        "Difference from Actual IBNR - Reported": col_diff(col12, col10)
    })

    col14 = pd.DataFrame({
        "Difference from Actual IBNR - Paid": col_diff(col12, col11)
    })


    cols = [col1, col2, col3, col4, col5, col6, col7, col8, col9, col10, col11, col12, col13, col14]

    results = pd.concat(cols, axis=1)

    format_dict = {
        # Dollar amounts
        "Claims at 12/31/2008 - Reported": "{:,.0f}",
        "Claims at 12/31/2008 - Paid": "{:,.0f}",
        "Case Outstanding": "{:,.0f}",
        "Projected Ult. Claims Using Dev. Method - Reported": "{:,.0f}",
        "Projected Ult. Claims Using Dev. Method - Paid": "{:,.0f}",
        "Estimated IBNR Using Dev. Method - Reported": "{:,.0f}",
        "Estimated IBNR Using Dev. Method - Paid": "{:,.0f}",
        "Actual IBNR": "{:,.0f}",
        "Difference from Actual IBNR - Reported": "{:,.0f}",
        "Difference from Actual IBNR - Paid": "{:,.0f}",

        # Ages
        "Age of Accident Year at 12/31/08": "{:.0f}",

        # Factors
        "CDF to Ult. - Reported": "{:.3f}",
        "CDF to Ult. - Paid": "{:.3f}",
    }

    results = results.set_index("Accident Year")

    total_cols = [
        "Claims at 12/31/2008 - Reported",
        "Claims at 12/31/2008 - Paid",
        "Case Outstanding",
        "Projected Ult. Claims Using Dev. Method - Reported",
        "Projected Ult. Claims Using Dev. Method - Paid",
        "Estimated IBNR Using Dev. Method - Reported",
        "Estimated IBNR Using Dev. Method - Paid",
        "Actual IBNR",
        "Difference from Actual IBNR - Reported",
        "Difference from Actual IBNR - Paid",
    ]
    
    results.loc["Total", total_cols] = results[total_cols].sum()

    display(HTML(f"<h3>Exhibit IV Sheet 6 — {scenario}</h3>"))

    check_cols = [
        "Projected Ult. Claims Using Dev. Method - Reported",
        "Projected Ult. Claims Using Dev. Method - Paid",
        "Estimated IBNR Using Dev. Method - Reported",
        "Estimated IBNR Using Dev. Method - Paid",
        "Actual IBNR",
        "Difference from Actual IBNR - Reported",
        "Difference from Actual IBNR - Paid",
    ]

    #print(results[check_cols].to_string())


    display(
        results.style.format(format_dict, na_rep="")
    )

    

    return results


Finally, we generate the tables in Sheet 6 calling the function once for each scenario:

In [0]:
res_1 = Ex4Sht6("Steady State", data)

res_2 = Ex4Sht6("Changing Product Mix", data)


In [0]:
estimate_cols = [
    "Projected Ult. Claims Using Dev. Method - Reported",
    "Projected Ult. Claims Using Dev. Method - Paid",
    "Estimated IBNR Using Dev. Method - Reported",
    "Estimated IBNR Using Dev. Method - Paid",
]

assert np.allclose(
    res_1.loc[
        res_1.index != "Total",
        estimate_cols,
    ].values,
    np.array([
    [1500000, 1500000,       0,       0],
    [1575000, 1575000,       0,       0],
    [1653750, 1653750,       0,       0],
    [1736438, 1736438,       0,       0],
    [1823259, 1823259,    8509,    8509],
    [1914422, 1914422,   29354,   29354],
    [2010143, 2010143,   61644,   61644],
    [2110651, 2110651,  173073,  173073],
    [2216183, 2216183,  363454,  363454],
    [2326992, 2326992,  758599,  758599],
]),
    atol=1,
    rtol=1e-5,
)

assert np.allclose(
    res_2.loc[
        res_2.index != "Total",
        estimate_cols,
    ].values,
    np.array([
    [1500000, 1500000,       0,       0],
    [1575000, 1575000,       0,       0],
    [1653750, 1653750,       0,       0],
    [1736438, 1736438,       0,       0],
    [1823259, 1823259,    8509,    8509],
    [1914422, 1914422,   29354,   29354],
    [2262942, 2251655,   69397,   58110],
    [2693735, 2650749,  222289,  179303],
    [3217775, 3091666,  537288,  411179],
    [3842645, 3592939, 1285950, 1036245],
]),
    atol=1,
    rtol=1e-5,
)